In [ ]:
"""
Word Sense Disambiguation using Lesk's Algorithm
=================================================

This program implements the Lesk Algorithm for Word Sense Disambiguation (WSD)
using NLTK and Princeton's WordNet lexical database.

Author: NLP Lab Submission
Date: February 2026

Description:
    The Lesk Algorithm disambiguates word senses by comparing the overlap
    between the context of an ambiguous word and the dictionary definitions
    (glosses) of each of its possible senses in WordNet.
"""

import nltk
from nltk.corpus import wordnet as wn
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string


# ============================================================================
# STEP 1: Download Required NLTK Resources
# ============================================================================

def download_nltk_resources():
    """
    Automatically downloads the required NLTK resources.
    
    Resources needed:
    - wordnet: Princeton's WordNet lexical database
    - punkt: Tokenizer models for sentence/word tokenization
    - omw-1.4: Open Multilingual Wordnet (required for WordNet 3.0+)
    - stopwords: Common stopwords for filtering
    """
    print("="*60)
    print("Downloading required NLTK resources...")
    print("="*60)
    
    resources = ['wordnet', 'punkt', 'omw-1.4', 'stopwords', 'punkt_tab']
    
    for resource in resources:
        try:
            nltk.download(resource, quiet=True)
            print(f"  [✓] {resource} downloaded successfully")
        except Exception as e:
            print(f"  [✗] Error downloading {resource}: {e}")
    
    print("="*60)
    print("NLTK resources ready!\n")


# ============================================================================
# STEP 2: Preprocessing Functions
# ============================================================================

def preprocess_text(text):
    """
    Preprocesses the input text by:
    1. Converting to lowercase
    2. Tokenizing into individual words
    3. Removing punctuation
    4. Removing stopwords (common words like 'the', 'is', 'a')
    
    Args:
        text (str): The input text to preprocess
    
    Returns:
        set: A set of cleaned, meaningful words
    """
    # Convert to lowercase for case-insensitive comparison
    text = text.lower()
    
    # Tokenize the text into individual words
    tokens = word_tokenize(text)
    
    # Get English stopwords
    stop_words = set(stopwords.words('english'))
    
    # Remove punctuation and stopwords
    # Keep only alphabetic tokens that are not stopwords
    cleaned_tokens = [
        token for token in tokens
        if token.isalpha() and token not in stop_words
    ]
    
    return set(cleaned_tokens)


def get_all_senses(word):
    """
    Retrieves all possible senses (synsets) of a word from WordNet.
    
    Args:
        word (str): The ambiguous word to look up
    
    Returns:
        list: A list of Synset objects representing all possible meanings
    """
    return wn.synsets(word)


# ============================================================================
# STEP 3: Signature Extraction Functions
# ============================================================================

def get_sense_signature(sense):
    """
    Extracts the signature (collection of related words) for a given sense.
    
    The signature includes words from:
    a) The definition (gloss) of the sense
    b) Example sentences of the sense
    c) Lemma names (synonyms)
    d) Hypernym definitions (more general concepts)
    
    Args:
        sense: A WordNet Synset object
    
    Returns:
        set: A set of words forming the sense's signature
    """
    signature = set()
    
    # a) Get words from the sense definition (gloss)
    definition = sense.definition()
    signature.update(preprocess_text(definition))
    
    # b) Get words from example sentences
    for example in sense.examples():
        signature.update(preprocess_text(example))
    
    # c) Get lemma names (synonyms) - these are the actual word forms
    for lemma in sense.lemmas():
        # Lemma names may contain underscores (e.g., "bank_deposit")
        # Split and add individual words
        lemma_words = lemma.name().replace('_', ' ').lower()
        signature.update(preprocess_text(lemma_words))
    
    # d) Get hypernym definitions (more general concepts)
    # Hypernyms help capture semantic relationships
    for hypernym in sense.hypernyms():
        hypernym_def = hypernym.definition()
        signature.update(preprocess_text(hypernym_def))
        
        # Also add hypernym lemma names
        for lemma in hypernym.lemmas():
            lemma_words = lemma.name().replace('_', ' ').lower()
            signature.update(preprocess_text(lemma_words))
    
    return signature


# ============================================================================
# STEP 4: Core Lesk Algorithm Implementation
# ============================================================================

def calculate_overlap(context_signature, sense_signature):
    """
    Calculates the overlap score between context and sense signatures.
    
    The overlap is simply the count of common words between the two sets.
    A higher overlap indicates a better match.
    
    Args:
        context_signature (set): Preprocessed words from the context sentence
        sense_signature (set): Signature words from a WordNet sense
    
    Returns:
        int: The number of common words (overlap score)
    """
    # Find the intersection of both sets
    common_words = context_signature.intersection(sense_signature)
    return len(common_words), common_words


def lesk_algorithm(sentence, ambiguous_word):
    """
    Implements the Lesk Algorithm for Word Sense Disambiguation.
    
    Algorithm Steps:
    1. Get all possible senses of the ambiguous word from WordNet
    2. Preprocess the context sentence to get context signature
    3. For each sense:
       - Extract the sense signature (definition, examples, lemmas, hypernyms)
       - Calculate overlap between context and sense signature
    4. Select the sense with maximum overlap
    
    Args:
        sentence (str): The context sentence containing the ambiguous word
        ambiguous_word (str): The word to disambiguate
    
    Returns:
        tuple: (best_sense, all_scores) or (None, None) if word not found
    """
    # Step 1: Get all possible senses from WordNet
    senses = get_all_senses(ambiguous_word)
    
    # Handle case when word is not found in WordNet
    if not senses:
        return None, None
    
    # Step 2: Preprocess the context sentence
    # Remove the ambiguous word itself from context to avoid self-matching
    context_signature = preprocess_text(sentence)
    context_signature.discard(ambiguous_word.lower())
    
    # Step 3: Calculate overlap for each sense
    sense_scores = []
    
    for sense in senses:
        # Get the signature for this sense
        sense_signature = get_sense_signature(sense)
        
        # Calculate overlap score
        overlap_score, common_words = calculate_overlap(context_signature, sense_signature)
        
        # Store the sense, score, and common words for later display
        sense_scores.append({
            'sense': sense,
            'score': overlap_score,
            'common_words': common_words,
            'definition': sense.definition()
        })
    
    # Step 4: Find the sense with maximum overlap
    # If there's a tie, the first sense is selected (usually most frequent)
    best_sense_info = max(sense_scores, key=lambda x: x['score'])
    
    return best_sense_info, sense_scores


# ============================================================================
# STEP 5: Display Functions
# ============================================================================

def display_results(ambiguous_word, best_sense_info, all_scores):
    """
    Displays the disambiguation results in a clear, formatted manner.
    
    Shows:
    - All possible senses with their definitions
    - Overlap score for each sense
    - The selected best sense
    - Definition of the selected sense
    
    Args:
        ambiguous_word (str): The word that was disambiguated
        best_sense_info (dict): Information about the selected best sense
        all_scores (list): List of all senses with their scores
    """
    print("\n" + "="*70)
    print(f"  WORD SENSE DISAMBIGUATION RESULTS FOR: '{ambiguous_word.upper()}'")
    print("="*70)
    
    # Display all possible senses
    print(f"\n{'─'*70}")
    print("  ALL POSSIBLE SENSES FROM WORDNET:")
    print(f"{'─'*70}")
    
    for i, score_info in enumerate(all_scores, 1):
        sense = score_info['sense']
        print(f"\n  Sense {i}: {sense.name()}")
        print(f"  Definition: {score_info['definition']}")
        
        # Show examples if available
        if sense.examples():
            print(f"  Examples: {'; '.join(sense.examples()[:2])}")
        
        print(f"  Overlap Score: {score_info['score']}")
        
        if score_info['common_words']:
            print(f"  Matching Words: {', '.join(score_info['common_words'])}")
        else:
            print("  Matching Words: (none)")
    
    # Display the selected sense
    print(f"\n{'─'*70}")
    print("  DISAMBIGUATION RESULT:")
    print(f"{'─'*70}")
    
    best_sense = best_sense_info['sense']
    print(f"\n  ★ Selected Sense: {best_sense.name()}")
    print(f"  ★ Part of Speech: {get_pos_name(best_sense.pos())}")
    print(f"  ★ Definition: {best_sense_info['definition']}")
    print(f"  ★ Overlap Score: {best_sense_info['score']}")
    
    if best_sense_info['common_words']:
        print(f"  ★ Matching Context Words: {', '.join(best_sense_info['common_words'])}")
    
    print("\n" + "="*70)


def get_pos_name(pos_tag):
    """
    Converts WordNet POS tags to human-readable names.
    
    Args:
        pos_tag (str): Single character POS tag from WordNet
    
    Returns:
        str: Human-readable part-of-speech name
    """
    pos_map = {
        'n': 'Noun',
        'v': 'Verb',
        'a': 'Adjective',
        's': 'Adjective Satellite',
        'r': 'Adverb'
    }
    return pos_map.get(pos_tag, 'Unknown')


# ============================================================================
# STEP 6: Main Interactive Program
# ============================================================================

def main():
    """
    Main function that runs the interactive Word Sense Disambiguation program.
    
    The program:
    1. Downloads required NLTK resources
    2. Prompts user for a sentence and ambiguous word
    3. Runs the Lesk algorithm
    4. Displays the results
    5. Allows multiple queries
    """
    # Download required resources
    download_nltk_resources()
    
    # Display program header
    print("\n" + "="*70)
    print("       WORD SENSE DISAMBIGUATION USING LESK'S ALGORITHM")
    print("                 (Custom Implementation)")
    print("="*70)
    print("\nThis program disambiguates word meanings using context overlap.")
    print("Enter 'quit' or 'exit' to end the program.\n")
    
    while True:
        try:
            # Get user input
            print("-"*70)
            sentence = input("\nEnter a sentence containing an ambiguous word:\n>>> ").strip()
            
            # Check for exit command
            if sentence.lower() in ['quit', 'exit', 'q']:
                print("\nThank you for using the WSD program. Goodbye!")
                break
            
            # Validate sentence input
            if not sentence:
                print("Error: Please enter a valid sentence.")
                continue
            
            ambiguous_word = input("\nEnter the ambiguous word to disambiguate:\n>>> ").strip()
            
            # Check for exit command
            if ambiguous_word.lower() in ['quit', 'exit', 'q']:
                print("\nThank you for using the WSD program. Goodbye!")
                break
            
            # Validate word input
            if not ambiguous_word:
                print("Error: Please enter a valid word.")
                continue
            
            # Check if the word is in the sentence
            if ambiguous_word.lower() not in sentence.lower():
                print(f"Warning: The word '{ambiguous_word}' was not found in the sentence.")
                print("Proceeding with disambiguation anyway...\n")
            
            # Run the Lesk algorithm
            best_sense_info, all_scores = lesk_algorithm(sentence, ambiguous_word)
            
            # Handle case when word is not found in WordNet
            if best_sense_info is None:
                print(f"\n❌ Error: The word '{ambiguous_word}' was not found in WordNet.")
                print("   Please try a different word or check the spelling.")
                continue
            
            # Display results
            display_results(ambiguous_word, best_sense_info, all_scores)
            
        except KeyboardInterrupt:
            print("\n\nProgram interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ An error occurred: {e}")
            print("   Please try again with different input.")


# ============================================================================
# STEP 7: Algorithm Explanation
# ============================================================================

def print_algorithm_explanation():
    """
    Prints a detailed explanation of how the Lesk Algorithm works.
    """
    explanation = """
╔══════════════════════════════════════════════════════════════════════╗
║              HOW THE LESK ALGORITHM WORKS                            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  The Lesk Algorithm is a classic dictionary-based approach to        ║
║  Word Sense Disambiguation (WSD), proposed by Michael Lesk in 1986.  ║
║                                                                      ║
║  BASIC PRINCIPLE:                                                    ║
║  ----------------                                                    ║
║  The algorithm assumes that words in a sentence relate to the same   ║
║  topic. Therefore, the correct sense of an ambiguous word should     ║
║  have a dictionary definition that shares the most words with the    ║
║  definitions of neighboring context words.                           ║
║                                                                      ║
║  ALGORITHM STEPS:                                                    ║
║  ----------------                                                    ║
║  1. INPUT: A sentence and an ambiguous word within it                ║
║                                                                      ║
║  2. PREPROCESSING:                                                   ║
║     - Tokenize the sentence into individual words                    ║
║     - Remove stopwords (common words like 'the', 'is', 'a')          ║
║     - Convert to lowercase for uniformity                            ║
║                                                                      ║
║  3. SENSE RETRIEVAL:                                                 ║
║     - Look up all possible senses of the ambiguous word in WordNet   ║
║                                                                      ║
║  4. SIGNATURE CREATION:                                              ║
║     For each sense, create a "signature" by extracting words from:   ║
║     a) The definition (gloss) of the sense                           ║
║     b) Example sentences provided in WordNet                         ║
║     c) Lemma names (synonyms of the word)                            ║
║     d) Hypernym definitions (more general concepts)                  ║
║                                                                      ║
║  5. OVERLAP CALCULATION:                                             ║
║     - Compare the context words with each sense's signature          ║
║     - Count the number of common words (overlap)                     ║
║                                                                      ║
║  6. SENSE SELECTION:                                                 ║
║     - Choose the sense with the highest overlap score                ║
║     - In case of a tie, prefer the most frequent sense               ║
║                                                                      ║
║  EXAMPLE:                                                            ║
║  --------                                                            ║
║  Sentence: "I went to the bank to deposit my money."                 ║
║  Ambiguous word: "bank"                                              ║
║                                                                      ║
║  Possible senses:                                                    ║
║  - bank (financial institution)                                      ║
║  - bank (river bank)                                                 ║
║                                                                      ║
║  Context words: {went, deposit, money}                               ║
║                                                                      ║
║  The financial sense's signature includes words like:                ║
║  {deposit, money, financial, institution, account, ...}              ║
║                                                                      ║
║  The river sense's signature includes words like:                    ║
║  {slope, water, river, edge, shore, ...}                             ║
║                                                                      ║
║  Overlap with financial sense: {deposit, money} = 2                  ║
║  Overlap with river sense: {} = 0                                    ║
║                                                                      ║
║  Result: Financial sense is selected (higher overlap)                ║
║                                                                      ║
║  LIMITATIONS:                                                        ║
║  ------------                                                        ║
║  - Relies on dictionary definitions being comprehensive              ║
║  - May fail when context words don't appear in definitions           ║
║  - Doesn't consider word order or syntactic relationships            ║
║  - Extended versions (like this implementation) use hypernyms        ║
║    and examples to improve accuracy                                  ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
"""
    print(explanation)


# ============================================================================
# Program Entry Point
# ============================================================================

if __name__ == "__main__":
    # Print algorithm explanation first
    print_algorithm_explanation()
    
    # Run the main interactive program
    main()
